In [ ]:
####  This Code is used to pull in all of the extracted river profiles (snapped to SWORD nodes), filter them for width, clean up their columns, and combine them into a single dataframe ####
## My data was located in 3 locations: inital data pulled for L8, L8 Updates, and L9 data ##

In [ ]:
# Import Libraries
import pandas as pd
import geopandas as gpd
import glob
import os
import numpy as np
import scipy.stats as stats
from tqdm.notebook import tqdm

In [ ]:
#############################################
########### Pull in the initial datasets ############
############################################

In [ ]:
###  Pull in the Dam File ###
Dams = gpd.read_file(r"F:\Insert_File_Path_of_Shapefile\Study_Dams.shp") # Update this file path
Dams_List  = Dams["grod_id"].tolist()
Dams_List.sort()

In [ ]:
######## My data was split across 3 folders. So this process is repeated several times. Remove cells as needed ########

In [ ]:
#### Pull in Original Data ####

### Pull in the Temperature Data -- Up/Ds FIltering ###
# Set up the location of the data
FilePath_OG = r"F:\Insert_File_Path_of_Temperature_CSVs_from_GEE_Snapped_to_SWORD_Nodes_L8" # Update this file path

# Get the List of Monthly Data
CSVFiles_OG = glob.glob(os.path.join(FilePath_OG, "*Avg_Img_Temps.csv")) 

# Loop through the files for each dam and make one dataframe
All_Mo_Avg_OG = pd.DataFrame()
for i in tqdm(range(len(CSVFiles_OG))):
    try:
        x = pd.read_csv(CSVFiles_OG[i], engine='python')
        All_Mo_Avg_OG = pd.concat([All_Mo_Avg_OG,x],axis=0)
    except pd.errors.EmptyDataError:
        print(CSVFiles_OG[i], " is empty and has been skipped.") # In case some of the CSV files are empty

## Looking at just the wide points, and non dam points
All_Mo_Avg_OG_Subset = All_Mo_Avg_OG[(All_Mo_Avg_OG['Avg_RWC_Wid'] >= 100) & (All_Mo_Avg_OG['Up_Ds']!= 'Dam')]

### Get Dam Info on the nodes/temperature Values  and clean up the dataset ##
HydroPower_OG = pd.merge(All_Mo_Avg_OG_Subset, Dams[['grod_id', 'dataset']], left_on = 'Assgn_dam', right_on = 'grod_id' , how='left').drop(columns = [ 'grod_id'])

# Remove Unwanted Columns
Dams_Temp_Clean_1 = HydroPower_OG.drop(columns=['Unnamed: 0'])

# Make an easier code for hydropower variables (joined from HILARRI Database) -- Updating the dictionary for simplicity
damtype_dict = {'Hydropower dam associated with power plant; no reservoir': 'HDNR', 'Hydropower dam associated with reservoir and power plant': 'HDR', 'Hydropower dam only; no reservoir or power plant': 'HDNR','Hydropower dam associated with reservoir; no power plant':'HDR', None: 'NoHydro'} # dictionary
Dams_Temp_Clean_1['HydroCode'] = Dams_Temp_Clean_1['dataset'].map(damtype_dict)  # apply dictionary

# Group datata into distance bins
bins = np.arange(-100, 100, 2).tolist() # create 2km bins
Dams_Temp_Clean_1['Bins'] = pd.cut(Dams_Temp_Clean_1['Dam_Dist_km'], bins) # apply bins

# Create a Code to easily group based on Up/Ds and Lake Flag
Dams_Temp_Clean_1['Up_Ds_Grp']  = np.nan
Dams_Temp_Clean_1['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_1['Up_Ds'] == 'Downstream') & (Dams_Temp_Clean_1['lakeflag'] == 0) , 'Downstream River', Dams_Temp_Clean_1['Up_Ds_Grp'])
Dams_Temp_Clean_1['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_1['Up_Ds'] == 'Downstream') & (Dams_Temp_Clean_1['lakeflag'] > 0) , 'Downstream Reservoir', Dams_Temp_Clean_1['Up_Ds_Grp'])
Dams_Temp_Clean_1['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_1['Up_Ds'] == 'Upstream') & (Dams_Temp_Clean_1['lakeflag'] == 0) , 'Upstream River', Dams_Temp_Clean_1['Up_Ds_Grp'])
Dams_Temp_Clean_1['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_1['Up_Ds'] == 'Upstream') & (Dams_Temp_Clean_1['lakeflag'] > 0) , 'Upstream Reservoir', Dams_Temp_Clean_1['Up_Ds_Grp'])

# Preview
Dams_Temp_Clean_1.head()

In [ ]:
#### Pull in the L8 updates ####

### Pull in the Temperature Data -- Up/Ds FIltering ###
# Set up the location of the data
FilePath_L8_upd = r"F:\Insert_File_Path_of_Temperature_CSVs_from_GEE_Snapped_to_SWORD_Nodes_L8_Updates" # Update this file path"

# Get the List of Monthly Data
CSVFiles_L8upd = glob.glob(os.path.join(FilePath_L8_upd, "*Avg_Img_Temps.csv")) 

# Loop through the files for each dam and make one dataframe
All_Mo_Avg_L8upd = pd.DataFrame()
for i in tqdm(range(len(CSVFiles_L8upd))):
    try:
        x = pd.read_csv(CSVFiles_L8upd[i], engine='python')
        All_Mo_Avg_L8upd = pd.concat([All_Mo_Avg_L8upd,x],axis=0)
    except pd.errors.EmptyDataError:
        print(CSVFiles_L8upd[i], " is empty and has been skipped.") # In case some of the CSV files are empty

## Looking at just the wide points, and non dam points
All_Mo_Avg_L8upd_Subset = All_Mo_Avg_L8upd[(All_Mo_Avg_L8upd['Avg_RWC_Wid'] >= 100) & (All_Mo_Avg_L8upd['Up_Ds']!= 'Dam')]

### Get Dam Info on the nodes/temperature Values  and clean up the dataset ##
HydroPower_L8upd = pd.merge(All_Mo_Avg_L8upd_Subset, Dams[['grod_id', 'dataset']], left_on = 'Assgn_dam', right_on = 'grod_id' , how='left').drop(columns = [ 'grod_id'])

# Remove Unwanted Columns
Dams_Temp_Clean_2 = HydroPower_L8upd.drop(columns=['Unnamed: 0'])

# Make an easier code for hydropower variables -- Updating the dictionary for simplicity
Dams_Temp_Clean_2['HydroCode'] = Dams_Temp_Clean_2['dataset'].map(damtype_dict)  # apply dictionary

# Group datata into distance bins
bins = np.arange(-100, 100, 2).tolist() # create 2km bins
Dams_Temp_Clean_2['Bins'] = pd.cut(Dams_Temp_Clean_2['Dam_Dist_km'], bins) # apply bins

# Create a Code to easily group based on Up/Ds and Lake Flag
Dams_Temp_Clean_2['Up_Ds_Grp']  = np.nan
Dams_Temp_Clean_2['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_2['Up_Ds'] == 'Downstream') & (Dams_Temp_Clean_2['lakeflag'] == 0) , 'Downstream River', Dams_Temp_Clean_2['Up_Ds_Grp'])
Dams_Temp_Clean_2['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_2['Up_Ds'] == 'Downstream') & (Dams_Temp_Clean_2['lakeflag'] > 0) , 'Downstream Reservoir', Dams_Temp_Clean_2['Up_Ds_Grp'])
Dams_Temp_Clean_2['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_2['Up_Ds'] == 'Upstream') & (Dams_Temp_Clean_2['lakeflag'] == 0) , 'Upstream River', Dams_Temp_Clean_2['Up_Ds_Grp'])
Dams_Temp_Clean_2['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_2['Up_Ds'] == 'Upstream') & (Dams_Temp_Clean_2['lakeflag'] > 0) , 'Upstream Reservoir', Dams_Temp_Clean_2['Up_Ds_Grp'])

# Preview
Dams_Temp_Clean_2.head()

In [ ]:
#### Pull in the L9 Data ####

### Pull in the Temperature Data -- Up/Ds FIltering ###
# Set up the location of the data
FilePath_L9 = r"F:\Insert_File_Path_of_Temperature_CSVs_from_GEE_Snapped_to_SWORD_Nodes_L9" # Update this file path

# Get the List of Monthly Data
CSVFiles_L9 = glob.glob(os.path.join(FilePath_L9, "*Avg_Img_Temps.csv")) 

# Loop through the files for each dam and make one dataframe
All_Mo_Avg_L9 = pd.DataFrame()
for i in tqdm(range(len(CSVFiles_L9))):
    try:
        x = pd.read_csv(CSVFiles_L9[i], engine='python')
        All_Mo_Avg_L9 = pd.concat([All_Mo_Avg_L9,x],axis=0)
    except pd.errors.EmptyDataError:
        print(CSVFiles_L9[i], " is empty and has been skipped.") # In case some of the CSV files are empty

## Looking at just the wide points, and non dam points
All_Mo_Avg_L9_Subset = All_Mo_Avg_L9[(All_Mo_Avg_L9['Avg_RWC_Wid'] >= 100) & (All_Mo_Avg_L9['Up_Ds']!= 'Dam')]

### Get Dam Info on the nodes/temperature Values  and clean up the dataset ##
HydroPower_L9 = pd.merge(All_Mo_Avg_L9_Subset, Dams[['grod_id', 'dataset']], left_on = 'Assgn_dam', right_on = 'grod_id' , how='left').drop(columns = [ 'grod_id'])

# Remove Unwanted Columns
Dams_Temp_Clean_3 = HydroPower_L9.drop(columns=['Unnamed: 0'])

# Make an easier code for hydropower variables -- Updating the dictionary for simplicity
Dams_Temp_Clean_3['HydroCode'] = Dams_Temp_Clean_3['dataset'].map(damtype_dict)  # apply dictionary

# Group datata into distance bins
bins = np.arange(-100, 100, 2).tolist() # create 2km bins
Dams_Temp_Clean_3['Bins'] = pd.cut(Dams_Temp_Clean_3['Dam_Dist_km'], bins) # apply bins

# Create a Code to easily group based on Up/Ds and Lake Flag
Dams_Temp_Clean_3['Up_Ds_Grp']  = np.nan
Dams_Temp_Clean_3['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_3['Up_Ds'] == 'Downstream') & (Dams_Temp_Clean_3['lakeflag'] == 0) , 'Downstream River', Dams_Temp_Clean_3['Up_Ds_Grp'])
Dams_Temp_Clean_3['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_3['Up_Ds'] == 'Downstream') & (Dams_Temp_Clean_3['lakeflag'] > 0) , 'Downstream Reservoir', Dams_Temp_Clean_3['Up_Ds_Grp'])
Dams_Temp_Clean_3['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_3['Up_Ds'] == 'Upstream') & (Dams_Temp_Clean_3['lakeflag'] == 0) , 'Upstream River', Dams_Temp_Clean_3['Up_Ds_Grp'])
Dams_Temp_Clean_3['Up_Ds_Grp'] = np.where((Dams_Temp_Clean_3['Up_Ds'] == 'Upstream') & (Dams_Temp_Clean_3['lakeflag'] > 0) , 'Upstream Reservoir', Dams_Temp_Clean_3['Up_Ds_Grp'])

# Preview
Dams_Temp_Clean_3.head()

In [ ]:
### Combine the Files together ###
Dam_Temp_All_Obs = pd.concat([Dams_Temp_Clean_1, Dams_Temp_Clean_2, Dams_Temp_Clean_3], axis=0, ignore_index=True)

In [ ]:
## Save all the Combined Temps --- This File is used in the next notebook ## 
Dam_Temp_All_Obs.to_csv(r"F:\Insert_File_Output_Path_Here\Combined_Temps_Nodes.csv") # Update this file path